# 📗 Fine-tuning YOLO compost + ÉVAL COMPOST

Repart d'un `best.pt` **pré-entraîné** (produit par `colab_train.ipynb`, sur Drive), le
fine-tune sur **nos captures réelles**, et compare la perf compost **AVANT / APRÈS**.

Léger et rapide : tu peux relancer le fine-tuning autant de fois que tu veux (il repart
toujours du même pré-entraîné). **Renseigne `PRETRAIN_PATH` dans la cellule 3.**

In [ ]:
# 1. Clone du repo
BRANCH = 'centralisation'   # branche de travail ; mettre 'main' après fusion
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
%cd /content
!rm -rf /content/repo
!git clone --depth 1 --branch {BRANCH} https://{token}@github.com/TSResearch-hub/Compost_Waste_Yolo.git /content/repo
%cd /content/repo/compost-yolo

In [ ]:
# 2. Installation des dépendances
!pip install -q -e .

In [ ]:
# 3. Données : captures réelles -> split STRATIFIÉ (80% fine-tune / 20% test) + chemin du pré-entraîné
# >>> CHEMIN DU best.pt PRÉ-ENTRAÎNÉ (vu dans la sortie de colab_train.ipynb, sur Drive) : <<<
PRETRAIN_PATH = '/content/drive/MyDrive/compost/runs/train_XXXX/weights/best.pt'   # <-- À RENSEIGNER
COMPOST       = 'captures'
TEST_FRACTION = 0.2

import os, shutil, re, random
from pathlib import Path
from collections import Counter
import yaml
from google.colab import drive
if os.path.isdir('/content/drive') and not os.path.ismount('/content/drive'):
    shutil.rmtree('/content/drive', ignore_errors=True)
drive.mount('/content/drive')

z = f'/content/drive/MyDrive/compost/dataset_raw_{COMPOST}.zip'
assert os.path.exists(z), f'{z} introuvable sur Drive'
!cp {z} /content/
!unzip -q -o /content/dataset_raw_{COMPOST}.zip -d /content/dataset_raw_{COMPOST}

# split stratifié : TEST_FRACTION de CHAQUE session -> test ; reste -> fine-tune (seed fixe)
random.seed(42)
src = Path(f'/content/dataset_raw_{COMPOST}')
def session_of(stem):
    m = re.search(r'_(\d{2})_\d{2}_\d{2}', stem); h = int(m.group(1)) if m else 0
    return 's1.5' if h >= 13 else 's1'
by = {}
for img in (src/'images').glob('*'):
    if img.suffix.lower() in ('.jpg','.jpeg','.png'): by.setdefault(session_of(img.stem), []).append(img)
assign = {}
for s, imgs in by.items():
    random.shuffle(imgs); nt = round(len(imgs)*TEST_FRACTION)
    for i, im in enumerate(imgs): assign[im] = 'test' if i < nt else 'finetune'

# IMPORTANT : evaluate.py lit les labels dans  <path>/labels/<nom du dossier d'images>.
# On structure donc le TEST comme prepare_dataset : images/test + labels/test (data.yaml test='images/test').
# Le FINE-TUNE reste plat (images/ + labels/), il passe par prepare_dataset juste après.
dest = {
    'test':     ('/content/captures_test/images/test',  '/content/captures_test/labels/test'),
    'finetune': ('/content/captures_finetune/images',   '/content/captures_finetune/labels'),
}
for idst, ldst in dest.values():
    Path(idst).mkdir(parents=True, exist_ok=True); Path(ldst).mkdir(parents=True, exist_ok=True)
for img, sub in assign.items():
    idst, ldst = dest[sub]
    shutil.copy(img, f'{idst}/{img.name}')
    lf = src/'labels'/f'{img.stem}.txt'
    if lf.exists(): shutil.copy(lf, f'{ldst}/{lf.name}')

names = yaml.safe_load(open('configs/data.yaml'))['names']
Path('/content/captures_test/data.yaml').write_text(yaml.safe_dump(
    {'path':'/content/captures_test','train':'images/test','val':'images/test','test':'images/test',
     'names':dict(enumerate(names))}, allow_unicode=True, sort_keys=False))
!python scripts/prepare_dataset.py --source /content/captures_finetune --output /content/dataset_finetune
rec = Counter((session_of(i.stem), s) for i, s in assign.items())
for sub in ('finetune','test'):
    idst = dest[sub][0]
    print(f"captures {sub}: {len(list(Path(idst).glob('*')))} ("+", ".join(f"{s}:{rec[(s,sub)]}" for s in sorted(by))+")")

In [ ]:
# 3b. NOS DONNÉES (captures) — instances par classe + boîtes par image
import yaml
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
names = yaml.safe_load(open('configs/data.yaml'))['names']; x = range(len(names))
pc, bpi, n_img, n_neg = Counter(), [], 0, 0
src = Path('/content/dataset_raw_captures')
for img in (src/'images').glob('*'):
    if img.suffix.lower() not in ('.jpg','.jpeg','.png'): continue
    n_img += 1
    lf = src/'labels'/f'{img.stem}.txt'
    lines = [l for l in lf.read_text().splitlines() if l.strip()] if lf.exists() else []
    bpi.append(len(lines)); n_neg += (not lines)
    for l in lines: pc[int(l.split()[0])] += 1
fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
b = axes[0].bar(list(x), [pc[j] for j in x]); axes[0].bar_label(b, fmt='%d', padding=2)
if any(pc.values()): axes[0].set_yscale('log')
axes[0].set_xticks(list(x)); axes[0].set_xticklabels(names, rotation=20)
axes[0].set_title('Captures (nos données) — instances par classe'); axes[0].grid(axis='y', alpha=0.3)
cap=10; dist=Counter(min(k,cap) for k in bpi); xs=list(range(cap+1))
b2=axes[1].bar(xs,[dist[k] for k in xs]); axes[1].bar_label(b2, fmt='%d', padding=2)
axes[1].set_xticks(xs); axes[1].set_xticklabels([str(k) for k in xs[:-1]]+[f'{cap}+'])
axes[1].set_xlabel('boîtes/image'); axes[1].set_title('Captures — boîtes par image'); axes[1].grid(axis='y', alpha=0.3)
fig.tight_layout(); plt.show()
moy = sum(bpi)/len(bpi) if bpi else 0
print(f"Captures : {n_img} images dont {n_neg} négatives | {sum(pc.values())} instances | {moy:.2f} boîtes/image")
print("  ", ", ".join(f"{names[j]}:{pc[j]}" for j in x))

In [ ]:
# 4. ÉVAL B — COMPOST AVANT fine-tuning : le pré-entraîné sur le test held-out (baseline domain gap)
from pathlib import Path
PRE = Path(PRETRAIN_PATH)
assert PRE.exists(), f"PRETRAIN_PATH introuvable : {PRE} — renseigne-le (cellule 3)"
print('Pré-entraîné évalué sur le COMPOST :', PRE)
!python scripts/evaluate.py --weights {PRE} --data /content/captures_test/data.yaml --split test --runs-dir /content/runs
from IPython.display import Image, display
evals = sorted(Path('/content/runs').glob('eval_*test_*'), key=lambda p: p.stat().st_mtime)
if evals:
    print('--- confusion COMPOST (AVANT) ---')
    for img in sorted(evals[-1].rglob('*confusion*.png')): print(img.name); display(Image(str(img)))

In [ ]:
# 5. FINE-TUNING : repart du PRÉ-ENTRAÎNÉ et continue sur les captures (fine-tune split), lr BAS.
#  ⚠️ Tu peux RELANCER cette cellule (+ 4 et 6) en changeant --epochs / --lr0 : ça repart
#     TOUJOURS du même PRETRAIN_PATH, pas d'un fine-tune précédent.
!python scripts/train.py --model {PRETRAIN_PATH} --data /content/dataset_finetune/data.yaml \
    --epochs 30 --lr0 0.001 --run-prefix finetune \
    --runs-dir /content/runs \
    --backup-dir /content/drive/MyDrive/compost/backups --backup-every 10

In [ ]:
# 6. ÉVAL C — COMPOST APRÈS fine-tuning : le modèle fine-tuné sur le MÊME test held-out.
#    Compare directement à l'ÉVAL B (cellule 4) : le fine-tuning a-t-il aidé sur le vrai compost ?
from pathlib import Path
cands = list(Path('/content/runs').glob('finetune_*/weights/best.pt'))
cands += list(Path('/content/runs').glob('train_*/weights/best.pt'))   # anciens runs, avant le renommage
assert cands, "Aucun fine-tune trouvé dans /content/runs — lance la cellule 5 d'abord."
FINETUNED = sorted(cands, key=lambda p: p.stat().st_mtime)[-1]
print('Fine-tuné évalué sur le COMPOST :', FINETUNED)
!python scripts/evaluate.py --weights {FINETUNED} --data /content/captures_test/data.yaml --split test --runs-dir /content/runs
from IPython.display import Image, display
evals = sorted(Path('/content/runs').glob('eval_*test_*'), key=lambda p: p.stat().st_mtime)
if evals:
    print('--- confusion COMPOST (APRÈS) ---')
    for img in sorted(evals[-1].rglob('*confusion*.png')): print(img.name); display(Image(str(img)))

In [ ]:
# 7. Copie des runs de fine-tuning vers Drive
!mkdir -p /content/drive/MyDrive/compost/runs
!cp -r /content/runs/* /content/drive/MyDrive/compost/runs/
!ls /content/drive/MyDrive/compost/runs